# Ejercicio 5: Espacio Vectorial

## Objetivo de la práctica
- Implementar un Sistema de Recuperación de Información completo, desde la lectura del corpus hasta la recuperación de resultados.

## Parte 0: Carga del Corpus

Vamos a utilizar la API de Kaggle para acceder al dataset _Wikipedia Text Corpus for NLP and LLM Projects_

El corpus está disponible desde este [link](https://www.kaggle.com/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects?utm_source=chatgpt.com)

### Actividad

1. Carga el corpus
2. Realiza las etapas de preprocesamiento sobre el corpus


In [31]:
import kagglehub
import os
import pandas as pd
import re

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Carga Corpus
path = kagglehub.dataset_download(
    "gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects"
)

archivo_corpus = os.path.join(path, "wikipedia_text_corpus.csv")

df = pd.read_csv(archivo_corpus)

print("Corpus cargado.")
print("Tamaño inicial:", df.shape)
print("Columnas:", df.columns.tolist())

display(df.head())


df = df.dropna(subset=[TEXT_COL])
df = df.drop_duplicates(subset=[TEXT_COL])
df = df.reset_index(drop=True)

print("Tamaño después de eliminar nulos y duplicados:", df.shape)


# Preprocesamiento del texto

stop_words = set(ENGLISH_STOP_WORDS)

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    
    tokens = text.split()
    
    tokens = [
        token for token in tokens
        if token not in stop_words and len(token) > 2
    ]
    
    return tokens


df["tokens"] = df[TEXT_COL].apply(preprocess_text)
df["text_clean"] = df["tokens"].apply(lambda tokens: " ".join(tokens))

df = df[df["text_clean"].str.len() > 0]
df = df.reset_index(drop=True)

print("Tamaño final del corpus:", df.shape)

display(df[[TEXT_COL, "text_clean"]].head())

Corpus cargado.
Tamaño inicial: (10859, 2)
Columnas: ['Unnamed: 0', 'text']


,Unnamed: 0,text
0,1,Anovo\n\nAnovo (formerly A Novo) is a computer...
1,2,Battery indicator\n\nA battery indicator (also...
2,3,"Bob Pease\n\nRobert Allen Pease (August 22, 19..."
3,4,CAVNET\n\nCAVNET was a secure military forum w...
4,5,CLidar\n\nThe CLidar is a scientific instrumen...


Tamaño después de eliminar nulos y duplicados: (10859, 2)
Tamaño final del corpus: (10859, 4)


,text,text_clean
0,Anovo\n\nAnovo (formerly A Novo) is a computer...,anovo anovo novo computer services company bas...
1,Battery indicator\n\nA battery indicator (also...,battery indicator battery indicator known batt...
2,"Bob Pease\n\nRobert Allen Pease (August 22, 19...",bob pease robert allen pease august june analo...
3,CAVNET\n\nCAVNET was a secure military forum w...,cavnet cavnet secure military forum operationa...
4,CLidar\n\nThe CLidar is a scientific instrumen...,clidar clidar scientific instrument used measu...


## Parte 1: Recuperación con TF-IDF

### Actividad:
3. Obtén la representación vectorial de los documentos utilizando el modelo TF-IDF
4. A partir de un conjunto de 10 queries, verifica la recuperación del sistema

In [32]:
# Recuperación con TF-IDF

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Crear matriz TF-IDF
tfidf_vectorizer = TfidfVectorizer(max_features=50000)
tfidf_matrix = tfidf_vectorizer.fit_transform(df["text_clean"])

print("Matriz TF-IDF:", tfidf_matrix.shape)


queries = [
    "computer services company",
    "battery indicator device",
    "military communication network",
    "scientific instrument measurement",
    "american engineer biography",
    "software development company",
    "music band album",
    "university research education",
    "medical disease treatment",
    "political history government"
]


# recuperar documentos con TF-IDF
def search_tfidf(query, top_k=5):
    query_clean = " ".join(preprocess_text(query))
    
    query_vector = tfidf_vectorizer.transform([query_clean])
    scores = cosine_similarity(query_vector, tfidf_matrix).flatten()
    
    top_indices = scores.argsort()[::-1][:top_k]
    
    results = []
    
    for i, idx in enumerate(top_indices):
        results.append({
            "rank": i + 1,
            "doc_id": idx,
            "score": round(scores[idx], 4),
            "title": "Documento " + str(idx),
            "fragment": str(df.loc[idx, TEXT_COL])[:200]
        })
    
    return pd.DataFrame(results)


# Ejecutar las búsquedas con TF-IDF
tfidf_results = {}

for query in queries:
    print("\nConsulta:", query)
    
    result = search_tfidf(query, top_k=5)
    tfidf_results[query] = result
    
    display(result[["rank", "doc_id", "score", "title"]])

Matriz TF-IDF: (10859, 50000)

Consulta: computer services company


,rank,doc_id,score,title
0,1,10329,0.3517,Documento 10329
1,2,260,0.3516,Documento 260
2,3,5814,0.3316,Documento 5814
3,4,9489,0.3189,Documento 9489
4,5,8133,0.3186,Documento 8133



Consulta: battery indicator device


,rank,doc_id,score,title
0,1,1,0.5613,Documento 1
1,2,1391,0.4676,Documento 1391
2,3,4480,0.3274,Documento 4480
3,4,7039,0.2888,Documento 7039
4,5,4086,0.2765,Documento 4086



Consulta: military communication network


...

,rank,doc_id,score,title
0,1,931,0.4440,Documento 931
1,2,7479,0.4125,Documento 7479
2,3,1494,0.3742,Documento 1494
3,4,5945,0.3454,Documento 5945
4,5,8749,0.3329,Documento 8749



Consulta: medical disease treatment


,rank,doc_id,score,title
0,1,9736,0.4131,Documento 9736
1,2,3504,0.3530,Documento 3504
2,3,6731,0.3469,Documento 6731
3,4,712,0.3381,Documento 712
4,5,5615,0.3258,Documento 5615



Consulta: political history government


,rank,doc_id,score,title
0,1,6079,0.2695,Documento 6079
1,2,2285,0.2492,Documento 2285
2,3,3981,0.2453,Documento 3981
3,4,1061,0.2419,Documento 1061
4,5,1600,0.2328,Documento 1600


## Parte 2: Recuperación con BM25

### Actividad:
5. Implementa un sistema de recuperación usando el modelo BM25.
6. Para el mismo conjunto de 10 queries, verifica la recuperación del sistema

In [33]:
# Recuperación con BM25

import numpy as np
import math
from collections import Counter, defaultdict

# Preparar documentos tokenizados
corpus_tokens = df["tokens"].tolist()
N = len(corpus_tokens)

# Calcular longitud de documentos
doc_lengths = [len(doc) for doc in corpus_tokens]
avg_doc_length = sum(doc_lengths) / len(doc_lengths)

print("Número de documentos:", N)
print("Longitud promedio:", round(avg_doc_length, 2))


# Calcular frecuencias de términos
term_freqs = []
doc_freqs = defaultdict(int)

for doc in corpus_tokens:
    conteo = Counter(doc)
    term_freqs.append(conteo)
    
    for term in conteo:
        doc_freqs[term] += 1


# Calcular IDF
idf = {}

for term in doc_freqs:
    df_term = doc_freqs[term]
    idf[term] = math.log(1 + ((N - df_term + 0.5) / (df_term + 0.5)))


# Función BM25
def bm25_score(query_tokens, doc_index):
    k1 = 1.5
    b = 0.75
    score = 0
    
    doc_tf = term_freqs[doc_index]
    doc_len = doc_lengths[doc_index]
    
    for term in query_tokens:
        if term in doc_tf:
            tf = doc_tf[term]
            
            numerator = tf * (k1 + 1)
            denominator = tf + k1 * (1 - b + b * (doc_len / avg_doc_length))
            
            score += idf.get(term, 0) * (numerator / denominator)
    
    return score


# Buscar documentos con BM25
def search_bm25(query, top_k=5):
    query_tokens = preprocess_text(query)
    
    scores = []
    
    for i in range(N):
        scores.append(bm25_score(query_tokens, i))
    
    scores = np.array(scores)
    top_indices = scores.argsort()[::-1][:top_k]
    
    results = []
    
    for i, idx in enumerate(top_indices):
        results.append({
            "rank": i + 1,
            "doc_id": idx,
            "score": round(scores[idx], 4),
            "title": "Documento " + str(idx),
            "fragment": str(df.loc[idx, TEXT_COL])[:200]
        })
    
    return pd.DataFrame(results)


# Ejecutar BM25 con las mismas queries
bm25_results = {}

for query in queries:
    print("\nConsulta:", query)
    
    result = search_bm25(query, top_k=5)
    bm25_results[query] = result
    
    display(result[["rank", "doc_id", "score", "title"]])

Número de documentos: 10859
Longitud promedio: 414.36

Consulta: computer services company


,rank,doc_id,score,title
0,1,260,10.9901,Documento 260
1,2,10582,10.7444,Documento 10582
2,3,8645,10.3142,Documento 8645
3,4,8227,10.2254,Documento 8227
4,5,6985,9.7366,Documento 6985



Consulta: battery indicator device


,rank,doc_id,score,title
0,1,1,18.5602,Documento 1
1,2,7126,14.4245,Documento 7126
2,3,4086,12.6724,Documento 4086
3,4,3964,12.0834,Documento 3964
4,5,4106,11.6029,Documento 4106



Consulta: military communication network


...

,rank,doc_id,score,title
0,1,180,13.1163,Documento 180
1,2,6885,12.9811,Documento 6885
2,3,3392,12.6112,Documento 3392
3,4,9306,12.3432,Documento 9306
4,5,1244,12.3094,Documento 1244



Consulta: medical disease treatment


,rank,doc_id,score,title
0,1,10143,16.4302,Documento 10143
1,2,7675,16.3252,Documento 7675
2,3,4551,16.1506,Documento 4551
3,4,1767,15.4782,Documento 1767
4,5,232,15.3453,Documento 232



Consulta: political history government


,rank,doc_id,score,title
0,1,7619,14.4926,Documento 7619
1,2,8598,13.0939,Documento 8598
2,3,3066,12.9767,Documento 3066
3,4,3950,12.6013,Documento 3950
4,5,2285,11.8152,Documento 2285


## Parte 3: Comparación de resultados

### Actividad:
7. Verifica cuáles documentos son recuperados (y en qué orden) por cada modelo de recuperación 

In [35]:
# Evaluación

summary = []

for query in queries:
    print("\nConsulta:", query)
    
    tfidf_docs = tfidf_results[query]["doc_id"].tolist()
    bm25_docs = bm25_results[query]["doc_id"].tolist()
    
    common_docs = set(tfidf_docs).intersection(set(bm25_docs))
    
    comparison = pd.DataFrame({
        "rank": [1, 2, 3, 4, 5],
        "tfidf_doc_id": tfidf_docs,
        "bm25_doc_id": bm25_docs,
        "same_position": [
            tfidf_docs[i] == bm25_docs[i]
            for i in range(5)
        ]
    })
    
    display(comparison)
    
    print("Documentos en común:", sorted(list(common_docs)))
    print("Cantidad de documentos en común:", len(common_docs))
    
    summary.append({
        "query": query,
        "tfidf_ranking": tfidf_docs,
        "bm25_ranking": bm25_docs,
        "common_documents": sorted(list(common_docs)),
        "num_common_documents": len(common_docs)
    })


summary_df = pd.DataFrame(summary)

display(summary_df)


average_common = summary_df["num_common_documents"].mean()

print("Promedio de documentos en común:", round(average_common, 2))

if average_common >= 4:
    print("Los modelos recuperan documentos bastante parecidos.")
elif average_common >= 2:
    print("Los modelos coinciden en algunos documentos, pero también tienen diferencias.")
else:
    print("Los modelos recuperan documentos bastante diferentes.")



Consulta: computer services company


,rank,tfidf_doc_id,bm25_doc_id,same_position
0,1,10329,260,False
1,2,260,10582,False
2,3,5814,8645,False
3,4,9489,8227,False
4,5,8133,6985,False


Documentos en común: [260]
Cantidad de documentos en común: 1

Consulta: battery indicator device


,rank,tfidf_doc_id,bm25_doc_id,same_position
0,1,1,1,True
1,2,1391,7126,False
2,3,4480,4086,False
3,4,7039,3964,False
4,5,4086,4106,False


Documentos en común: [1, 4086]
Cantidad de documentos en común: 2

Consulta: military communication network


...

Documentos en común: []
Cantidad de documentos en común: 0

Consulta: political history government


,rank,tfidf_doc_id,bm25_doc_id,same_position
0,1,6079,7619,False
1,2,2285,8598,False
2,3,3981,3066,False
3,4,1061,3950,False
4,5,1600,2285,False


Documentos en común: [2285]
Cantidad de documentos en común: 1


,query,tfidf_ranking,bm25_ranking,common_documents,num_common_documents
0,computer services company,"[10329, 260, 5814, 9489, 8133]","[260, 10582, 8645, 8227, 6985]",[260],1
1,battery indicator device,"[1, 1391, 4480, 7039, 4086]","[1, 7126, 4086, 3964, 4106]","[1, 4086]",2
2,military communication network,"[2105, 790, 1889, 6014, 5330]","[2105, 3970, 932, 3690, 9166]",[2105],1
3,scientific instrument measurement,"[4735, 10829, 52, 2315, 1712]","[7086, 1712, 10829, 9336, 3708]","[1712, 10829]",2
4,american engineer biography,"[6187, 10280, 1650, 5660, 8750]","[6406, 7872, 8997, 1380, 8750]",[8750],1
5,software development company,"[824, 4982, 9315, 193, 1477]","[9350, 9219, 3722, 1791, 7909]",[],0
6,music band album,"[1579, 3852, 9826, 5365, 5024]","[7515, 10349, 1486, 5365, 10828]",[5365],1
7,university research education,"[931, 7479, 1494, 5945, 8749]","[180, 6885, 3392, 9306, 1244]",[],0
8,medical disease treatment,"[9736, 3504, 6731, 712, 5615]","[10143, 7675, 4551, 1767, 232]",[],0
9,political history government,"[6079, 2285, 3981, 1061, 1600]","[7619, 8598, 3066, 3950, 2285]",[2285],1


Promedio de documentos en común: 0.9
Los modelos recuperan documentos bastante diferentes.
